In [ ]:
import glob, os, sys, csv
import pandas as pd
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [ ]:
query = """
SELECT ACIDTRAB
    ,ASSISTMED
    ,CAUSABAS
    ,CIRCOBITO
    ,CIRURGIA
    ,CODMUNOCOR
    ,CODMUNRES
    ,DTNASC
    ,DTOBITO
    ,ESC
    ,ESCMAE
    ,ESTCIV
    ,EXAME
    ,FONTE
    ,GESTACAO
    ,GRAVIDEZ
    ,IDADE
    ,IDADEMAE
    ,LINHAA
    ,LINHAB
    ,LINHAC
    ,LINHAD
    ,LINHAII
    ,LOCOCOR
    ,NATURAL
    ,NECROPSIA
    ,OBITOGRAV
    ,OBITOPARTO
    ,OBITOPUERP
    ,OCUP
    ,OCUPMAE
    ,PARTO
    ,PESO
    ,QTDFILMORT
    ,QTDFILVIVO
    ,RACACOR
    ,SEXO
    ,TIPOBITO
  FROM mortalidade_temp"""

In [ ]:
pasta_arquivos = r"C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade"  # Altere para o caminho da sua pasta, se necessário

padrao_busca = os.path.join(pasta_arquivos, "Mortalidade_Geral_*.csv")
arquivos = sorted(glob.glob(padrao_busca))

for i, arquivo in enumerate(arquivos):
    print(f"Processando arquivo: {arquivo}")

    df_ = spark.read.csv(arquivo, header=True, inferSchema=True, sep=';')
    df_.createOrReplaceTempView("mortalidade_temp")
    df_cols = spark.sql(query)
    # Adiciona aS colunaS NOME_ARQ e ANO com base no nome do arquivo
    df_cols = \
        (df_cols.withColumns({"ANO": F.regexp_extract(F.lit(arquivo) , r"_(\d{4})(?:_|\.csv)", 1)
                             ,"NOME_ARQ": F.lit(arquivo)
                             }))

    df_cols.toPandas().to_csv(f"{pasta_arquivos}\\normalizar_colunas\\{os.path.basename(arquivo)}"
                             ,sep=';',encoding='utf-8'
                             ,quoting=csv.QUOTE_ALL
                             ,index=False)


    if i == 0:
        df_final = df_cols
    else:
        df_final = df_final.unionByName(df_cols)

    # if i >= 2:
    #     break

# df_cols.printSchema()

In [ ]:
df_final.filter("ANO >= '2020'").select("NOME_ARQ", "ANO").groupBy("NOME_ARQ", "ANO").count().limit(1000).show(truncate = False)